# Cosine similarity from scratch

For a query vector $q$ and document vector $x_i$, cosine similarity is

$$\operatorname{cos}(q, x_i) = \frac{q \cdot x_i}{\lVert q \rVert_2 \lVert x_i \rVert_2}. $$

The dot product measures alignment. Dividing by both L2 norms removes magnitude, so only direction remains. A zero vector is invalid because its norm makes the denominator zero. We normalize both sides, compute one matrix-vector product, sort by descending score, and use `chunk_id` ascending as a deterministic tie-break. If `k` exceeds the row count, the reference implementation returns all rows.

In [ ]:
import numpy as np

from ragbench.retrieval.dense import cosine_top_k

## Offline deterministic example

For $q=(1,1)$: the vector `(1, 1)` scores `1`; `(1, 0)` scores $1/\sqrt{2}$; and `(-1, 0)` scores $-1/\sqrt{2}$. The example uses no provider or database.

In [ ]:
query = np.array([1.0, 1.0])
matrix = np.array([[1.0, 0.0], [1.0, 1.0], [-1.0, 0.0]])
chunk_ids = ("axis", "same", "opposite")
hits = cosine_top_k(query, matrix, 3, chunk_ids)
[(hit.chunk_id, hit.score) for hit in hits]

## Stable ties and pgvector parity

Cosine distance in pgvector is `embedding <=> query`. Its matching similarity is `1 - distance`. SQL ordering uses distance ascending and then `chunk_id` ascending; NumPy uses score descending and then the same ID rule. This makes equal-score rankings independent of input or storage order. Configured CI may compare returned IDs and scores with absolute tolerance `1e-5`; this notebook does not claim the pending 50-query database run.

In [ ]:
tie_matrix = np.array([[1.0, 1.0], [1.0, -1.0]])
tie_hits = cosine_top_k(np.array([1.0, 0.0]), tie_matrix, 2, ("chunk-b", "chunk-a"))
assert [hit.chunk_id for hit in tie_hits] == ["chunk-a", "chunk-b"]